In [ ]:
from openai import OpenAI
import dspy

In [ ]:
import dspy
lm = dspy.LM('gpt-4.1-2025-04-14', api_key='', temperature = 1.5, cache=False)
dspy.configure(lm=lm)

In [ ]:
import dspy
from typing import List

class ExperiencedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Experienced (enacted) stigma*.

    Definition: Direct experiences of discrimination, exclusion, or negative treatment due to diabetes.

    Codebook Examples:
    - “I find a lot of people, they like to think of you as being the culprit…”
    - “The dietician was awful…she asked me if I exercise… ‘that’s not enough’…”
    - “His mother didn’t like me because I was diabetic… told him not to marry me…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing enacted stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing enacted stigma")

class PerceivedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Perceived (felt) stigma*.

    Definition: Awareness or belief that others hold negative attitudes or stereotypes about people with diabetes.

    Codebook Example:
    - “There’s this message that diabetes is this terrible thing that terrible people get because they do terrible things.”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing perceived stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing perceived stigma")

class AnticipatedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Anticipated stigma*.

    Definition: The expectation or fear of experiencing diabetes stigma in the future.

    Codebook Example:
    - “When I started getting older… I used to hide it from any boyfriends…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing anticipated stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing anticipated stigma")

class InternalisedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Internalised (self) stigma*.

    Definition: Acceptance and internalization of negative beliefs and stereotypes about oneself due to diabetes, leading to shame and self-blame.

    Codebook Examples:
    - “I felt guilty in the early days… it was my fault.”
    - “There is still somehow a feeling of stigma attached… you feel it’s your fault…”
    - “My brothers would never come and see me… they would say it was my own fault…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing internalised stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing internalised stigma")

class IntersectionalStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Intersectional stigma*.

    Definition: Diabetes stigma converging with other stigmatised conditions (e.g., obesity, schizophrenia) or characteristics (e.g., race, ethnicity).

    Codebook Example:
    - “Their appearance might be fat, they might look unhealthy and… that might be the main reason they’ve got a stigma rather than the diabetes.”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing intersectional stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing intersectional stigma")


# Data filtering

In [ ]:
import json
with open('/content/drive/MyDrive/stigma_project/final experiment results/train_data.json', 'r') as f:
  train_data = json.loads(f.read())
  f.close()

import json
with open('/content/drive/MyDrive/stigma_project/final experiment results/val_data.json', 'r') as f:
  val_data = json.loads(f.read())
  f.close()

all_data = train_data + val_data

In [ ]:
stigma_types = ['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

fewshot_samples = {x: [] for x in stigma_types}

for typ in stigma_types:
  for data_obj in all_data:
    if data_obj['label'] == 'yes stigma':
      if len(data_obj['types']) == 1 and typ in data_obj['types']:
        fewshot_samples[typ].append(data_obj)


In [ ]:
for k in fewshot_samples:
  print(k, '=', len(fewshot_samples[k]))

experienced stigma = 24
perceived stigma = 7
anticipated stigma = 3
internalized stigma = 53
intersectional stigma = 1


In [ ]:
type_generators = {'experienced stigma': dspy.Predict(ExperiencedStigmaPrompt),
                   'perceived stigma': dspy.Predict(PerceivedStigmaPrompt),
                   'anticipated stigma': dspy.Predict(AnticipatedStigmaPrompt),
                   'internalized stigma': dspy.Predict(InternalisedStigmaPrompt),
                   'intersectional stigma': dspy.Predict(IntersectionalStigmaPrompt)}

In [ ]:
import random

key = 'anticipated stigma'

ans = type_generators[key](examples=random.sample(fewshot_samples[key], min(3, len(fewshot_samples[key])))).new_post

In [ ]:
ans

"I'm starting a new semester soon and I'm honestly dreading the group projects. I'm always hesitant to tell classmates that I have diabetes because I'm worried they’ll judge me if I need to step out to treat a low or bring snacks to meetings. It just gets so tiring trying to be prepared, but also constantly afraid they'll see me as weak or unreliable..."

In [ ]:
generated_answers = {'experienced stigma': [],
                   'perceived stigma': [],
                   'anticipated stigma': [],
                   'internalized stigma': [],
                   'intersectional stigma': []}

for k in generated_answers:
  for _ in range(10):
    ans = type_generators[k](examples=random.sample(fewshot_samples[k], min(3, len(fewshot_samples[k])))).new_post
    generated_answers[k].append(ans)


In [ ]:
import json

with open('/content/drive/MyDrive/stigma_project/generated samples/gpt4.1-samples.json', 'w') as f:
  f.write(json.dumps(generated_answers))
  f.close()